# 13 | LangChain Embeddings：从文本向量化到语义检索

大模型应用里，聊天模型负责生成答案，嵌入模型负责把文本转换成向量。

向量可以表示文本的语义位置，因此常用于语义搜索、RAG 文档召回、相似问题匹配和长期记忆检索。

本文使用本地 Ollama 模型 `qwen3-embedding:latest`，演示 LangChain 中 Embeddings、VectorStore 和 Retriever 的最小闭环。

## 一、创建嵌入模型

`OllamaEmbeddings` 会把本地 Ollama 的 embedding 模型包装成 LangChain 标准 Embeddings 对象。

In [ ]:
from langchain_ollama import OllamaEmbeddings

# 创建本地 embedding 模型对象。
# 这一步只是初始化配置，还没有真正向模型发送文本。
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:latest",
    base_url="http://localhost:11434",
)

embeddings

## 二、向量化一个查询

`embed_query()` 用于向量化用户问题。返回值是一组浮点数，也就是向量。

In [ ]:
# 用户问题。
query = "LangChain 怎么保存长期记忆？"

# 把一个查询文本转换成向量。
query_vector = embeddings.embed_query(query)

print(type(query_vector))
print("向量维度:", len(query_vector))
print("前 8 个数字:", query_vector[:8])

向量维度由模型决定。实际项目中，同一个向量库必须使用同一个 embedding 模型，否则不同维度或不同向量空间的结果无法正确比较。

## 三、批量向量化文档

`embed_documents()` 用于向量化一组待检索文本。RAG 中通常先把知识库切成多个片段，再批量写入向量库。

In [ ]:
# 一个极简知识库。每个字符串代表一段文档片段。
docs = [
    "短期记忆使用 checkpointer 保存单个 thread 的临时对话状态，只在当前会话内生效。",
    "长期记忆用于跨会话记住用户信息；在 LangChain 和 LangGraph 中通常使用 store 保存跨 thread 共享的用户资料和业务事实。",
    "工具调用让 Agent 可以访问外部函数，例如查询天气、搜索网页或读取数据库。",
    "提示词模板可以把变量、系统指令和用户输入组合成模型需要的消息。",
]

# 把多段文本批量转换成向量。
doc_vectors = embeddings.embed_documents(docs)

print("文档数量:", len(doc_vectors))
print("第一条文档向量维度:", len(doc_vectors[0]))
print("第一条文档向量前 8 个数字:", doc_vectors[0][:8])

## 四、理解相似度计算

向量检索的核心是相似度计算。这里先手写一次余弦相似度，帮助理解 VectorStore 背后大致在做什么。

In [ ]:
import math


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个向量的余弦相似度。分数越高，语义越接近。"""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)


# 保存每篇文档和查询之间的相似度。
scores = []

# docs 和 doc_vectors 一一对应：docs[0] 对应 doc_vectors[0]。
for doc, doc_vector in zip(docs, doc_vectors):
    score = cosine_similarity(query_vector, doc_vector)
    scores.append((doc, score))

# 按相似度从高到低排序。
for doc, score in sorted(scores, key=lambda item: item[1], reverse=True):
    print(f"{score:.4f} | {doc}")

query是"LangChain 怎么保存长期记忆？"，所以第一条0.7910分的返回是合理的。

另外，embedding 检索不是关键词匹配，而是语义相似度比较的，所以排序结果会受到模型、查询写法、文档切块方式和文本长度等因素影响。

## 五、接入 VectorStore

真实应用中不需要手写相似度计算，通常把文本和向量交给向量库管理。这里使用 LangChain 内置的 `InMemoryVectorStore`，适合实验和小型 demo。

In [ ]:
import numpy as np
from langchain_core.vectorstores import InMemoryVectorStore

print("numpy 版本:", np.__version__)

# from_texts 会先调用 embeddings.embed_documents(docs)，
# 再把文本和向量保存到内存向量库中。
vectorstore = InMemoryVectorStore.from_texts(
    texts=docs,
    embedding=embeddings,
)

# 查询最相关的 2 条文档。
results = vectorstore.similarity_search(
    "跨会话记住用户信息应该用什么？",
    k=2,
)

for i, doc in enumerate(results, start=1):
    print(f"#{i}", doc.page_content)

注意：`InMemoryVectorStore` 的数据只存在当前 Python 进程中，重启后会丢失。

生产环境通常会换成 Postgres/pgvector、Milvus、Qdrant、Elasticsearch 等持久化向量库。

## 六、转换成 Retriever

Retriever 是 LangChain 中更通用的检索接口。很多 RAG 链路和 Agent 组件接收的是 retriever，而不是直接接收 vectorstore。

In [ ]:
# k=2 表示每次检索返回 2 条文档。
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

retrieved_docs = retriever.invoke("Agent 如何调用外部函数？")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"#{i}", doc.page_content)

## 七、最小 RAG 闭环

RAG 的基本流程是：先检索相关资料，再把资料放进提示词，最后交给聊天模型生成答案。下面先构造提示词。

In [ ]:
question = "LangChain 的长期记忆和短期记忆有什么区别？"

# 先检索相关上下文。
context_docs = retriever.invoke(question)

# 把检索结果拼成提示词上下文。
context = "\n".join(f"- {doc.page_content}" for doc in context_docs)

prompt = f"""
请只根据下面的上下文回答问题。

上下文：
{context}

问题：{question}
""".strip()

print(prompt)

如果需要生成最终回答，可以接入本地聊天模型。

embedding 模型负责召回资料，chat model 负责组织答案。

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("ollama:qwen2.5:14b")

answer = llm.invoke(prompt)
print(answer.content)

## 八、几个关键注意点

1. Embedding 模型和 Chat 模型不要混用。`qwen3-embedding` 用于向量化，`qwen3.6-plus`、`deepseek-v4-flash` 这类模型用于生成回答。
2. 不同Embedding 模型的能力不同，处理中文的时候，亲测qwen3-embedding要优于mxbai-embed-large。
3. 同一个向量库必须使用同一个 embedding 模型。换模型后，旧向量通常需要重建。
4. 文档切块会显著影响召回质量。真实项目中不要把大文档整段入库。
5. 查询写得越清楚，检索通常越稳定。必要时可以增加 metadata 过滤或 rerank。